# Hyperalignment from the Pattern-Alignment Perspective

## 1. Why Do We Need Hyperalignment?

Our brain produces a series of responses to a stimulus, and these responses should not be considered by focusing on only a single voxel or vertex. Instead, the responses of all voxels within a local cortical region at that moment should be combined to form a **response pattern**. In other words, the mental state at that moment is jointly represented by an activity pattern composed of a large number of voxels or vertices. Such a pattern can be viewed as a vector in a high-dimensional space: each voxel or vertex is one dimension, while a particular time point, stimulus, or condition corresponds to a complete pattern vector. Thus, when a person watches a movie, the many successive time points generate many pattern vectors, and these vectors together constitute that person's high-dimensional information space.

A pattern is meaningful not only because it is a series of voxel values, but also because it has stable relationships with other patterns. For example, if the patterns corresponding to moments A and B in a movie are very similar, while the patterns corresponding to moments A and C are very different, then these relationships—which patterns are similar to each other and which patterns are different from each other—constitute the information structure. Therefore, in this context, "information" is defined as the relationships among pattern vectors. Taken together, all of these pairwise relationships among patterns form the person's **representational geometry**.

For example, as illustrated in the figure below, for a given participant, if the neural representations of the two cat stimuli at the top are more similar to each other than either is to the cat stimulus at the bottom, the three stimuli would form a triangular configuration like this, with the two cats at the top positioned closer together.

```{image} pic/understand_ha/rg.png
:width: 1000px
```

Traditionally, how do we compare the representational patterns of two people in response to the same stimulus? We often use anatomical alignment: a standard template is constructed, and the functional information of all participants is aligned to this standard template. This process contains an implicit assumption: after alignment to the standard brain, location $(x, y, z)$ in participant A's brain represents the same information as location $(x, y, z)$ in participant B's brain.

The real problem, however, is that although different people's brains may represent the same information, the specific cortical pattern associated with that information is not identical across individuals. For example, when two people view the same moment in a movie, both form response vectors that can distinguish this moment from other moments. However, because fine-grained cortical topography differs across individuals, participant A's pattern may show strong responses in one group of voxels and weak responses in another group, whereas participant B may show a different spatial distribution. Therefore, even if traditional anatomical alignment aligns the anatomical locations of two brains as closely as possible, it cannot guarantee that the "same voxel location" carries the same informational function.

We therefore face the following situation: the patterns of different participants look different on the surface, but their internal pattern geometries are often highly similar. In other words, participant A and participant B use different cortical coordinate systems to construct the same information structure.

So, how Can We Establish Comparability? First, we know that the same information can be represented by different patterns, and that a pattern is composed of a large number of voxels or vertices. Therefore, we can no longer establish correspondence using individual voxels or vertices; instead, we should use the representational patterns jointly formed by a group of voxels or vertices.

Does this mean that we can align them arbitrarily, as long as the pattern representation points of each participant are aligned? No. As mentioned above, the actual "information" is reflected in the relationships among different pattern vectors—that is, in the representational geometry. Regardless of how the alignment is performed, a participant's representational geometry cannot change before and after the transformation.

Therefore, to ensure that the representational geometry remains unchanged, we need to apply the same rotation to all activity patterns. Aligning activity patterns is essentially a matter of recombining the values of the voxels or vertices. The goal of hyperalignment is therefore to find a transformation that linearly combines all voxels or vertices and uses this single transformation to align all pattern vectors simultaneously.

## 2. Procrustes Rotation: The Core Mechanism of Hyperalignment

Orthogonal Procrustes rotation (PR) estimates a rigid transformation that reorients one subject's activity patterns toward those of another subject or a common reference, without changing the internal geometry of that subject's response patterns.

PR updates each subject's activity patterns by linearly recombining the values across voxels or vertices, thereby aligning all of the subject's pattern vectors:

$$
\left\{
\begin{aligned}
\widehat{V}_1 &= w_{1(1)}V_1 + w_{2(1)}V_2 + \cdots + w_{n(1)}V_n, \\
\widehat{V}_2 &= w_{1(2)}V_1 + w_{2(2)}V_2 + \cdots + w_{n(2)}V_n, \\
&\ \vdots \\
\widehat{V}_n &= w_{1(n)}V_1 + w_{2(n)}V_2 + \cdots + w_{n(n)}V_n.
\end{aligned}
\right.
$$

<iframe src="2sub_pattern_align.html" title="Two-subject pattern alignment demo" style="width: 100%; aspect-ratio: 775 / 460; height: auto; border: 0; display: block;" loading="lazy"></iframe>


From the perspective of the mathematical operation itself, PR finds this single transformation by minimizing the Frobenius norm between two matrices. Formally, let

$$
X \in \mathbb{R}^{T \times V}
\quad \text{and} \quad
Y \in \mathbb{R}^{T \times V}
$$

denote two multivoxel response matrices measured over the same set of $T$ time points, with $V$ voxels or vertices. Here, $X$ may represent the data from one subject, while $Y$ may represent another subject or a reference common space. Orthogonal procrustes alignment seeks a transformation matrix

$$
R \in \mathbb{R}^{V \times V}
$$

that maps $X$ toward $Y$ by minimizing the squared Frobenius norm of the residual mismatch:

$$
\min_{R}\; \|XR - Y\|_F^2
\quad \text{subject to} \quad
R^\top R = I .
$$

Here, $\|\cdot\|_F$ denotes the Frobenius norm, which measures the overall discrepancy between the transformed source data $XR$ and the target data $Y$. The orthogonality constraint $R^\top R = I$ ensures that the transformation does not arbitrarily stretch, shrink, or shear the original data. Instead, it only reorients the voxel basis, thereby preserving the internal geometry of the source representational space.

This optimization problem has a closed-form solution based on singular value decomposition (SVD), which makes procrustes-based hyperalignment both computationally efficient and transparent at the matrix level.

## 3. Constructing a Common Representational Space

The goal of hyperalignment is not merely to align one subject to another, but to re-express individual neural response patterns in a **common representational space**. This space provides a shared coordinate system in which multivoxel activity patterns from different individuals can be compared, averaged, decoded, or modelled in a more meaningful way. By analogy with anatomical alignment, we can also understand that the information from a single subject includes uncontrollable noise and individual-specific components. If the data from all subjects were aligned to a single subject, that subject's noise and individual-specific components would also become targets that the other subjects' data need to approximate. Therefore, HA constructs a common space by iteratively performing orthogonal Procrustes rotation across subjects, so that the common space contains as little noise as possible while capturing as much of the shared components across subjects as possible.

Several strategies can be used to construct such a common representational space. The current `fMRI-HA` toolbox supports procrustes-based approaches, which iteratively align subject-specific response spaces to a shared template, as well as PCA-based methods, which derive a common low-dimensional structure from the group data. These alternatives allow users to choose an alignment strategy according to the goals and constraints of their analysis.

<iframe src="cspace_construct_pattern_align.html" title="Iterative common-space construction demo" style="width: 100%; aspect-ratio: 900 / 800; height: auto; border: 0; display: block;" loading="lazy"></iframe>


### Iterative Template Construction with Procrustes Rotation

```{image} pic/understand_ha/pr.png
:width: 1000px
```

In practice, a common representational space in procrustes-based hyperalignment is often constructed using an **iterative template algorithm**. Rather than treating a single subject as the final reference space, the algorithm progressively aligns individual subjects into a shared coordinate system and updates the group template based on the aligned data.

As illustrated in the schematic above, this procedure can be understood as a two-stage process.

**First stage: sequential alignment and template initialization.**  
One subject is first selected as an initial reference space. A second subject is aligned to this reference using procrustes rotation, and the aligned data are averaged with the reference data to form an interim template. Additional subjects are then aligned one by one to the current template, and the template is updated after each alignment by averaging the newly aligned data with the previously aligned data. After all subjects have been processed once, this procedure yields an initial, or first-pass, common template.

**Second stage: refinement using the first-pass template.**  
In the second pass, all subjects' original data are independently realigned to the first-pass template. The resulting aligned response matrices are then averaged across subjects to produce the final common representational space. This refinement step reduces potential bias introduced by the initial reference subject and produces a group-level template that is derived from all subjects rather than from any single individual.

Formally, let

$$
B_i \in \mathbb{R}^{T \times V}
$$

denote the response matrix of subject $i$, where $T$ is the number of matched time points or stimulus samples and $V$ is the number of voxels or features. Let

$$
R_i \in \mathbb{R}^{V \times V}
$$

denote the subject-specific orthogonal transformation estimated by procrustes alignment. Given a current template $M$, each subject-specific transformation can be estimated as

$$
R_i
=
\arg\min_{R}
\left\| B_i R - M \right\|_F^2
\quad
\text{subject to}
\quad
R^\top R = I .
$$

After aligning all subjects to the current template, the template is updated by averaging the transformed response matrices:

$$
M
=
\frac{1}{N}
\sum_{i=1}^{N}
B_i R_i .
$$

In an iterative implementation, these two steps are repeated: subject-specific transformations are estimated relative to the current template, and the template is then recomputed from the aligned data. The final common space therefore reflects the shared representational geometry supported by the group, while each subject retains a separate mapping into that space.

Importantly, although the initial coordinate system may be anchored by a reference subject, the final common model space is not simply that subject's native space. It is a group-derived representational template. Even the initial reference subject can receive a non-identity transformation during refinement, because the final template is recomputed from the aligned data of all subjects.

### Common Space Construction with PCA

```{image} pic/understand_ha/pca.png
:width: 600px
```

In addition to procrustes-based iterative alignment, a common representational space can also be constructed using a **PCA-based approach**. Instead of building the template through sequential subject-to-template alignment, this approach estimates shared structure by applying dimensionality reduction to the group data. The resulting components capture dominant patterns of variance across subjects and provide a group-derived basis for constructing a common template.

A key advantage of the PCA-based approach is that all subjects contribute to the decomposition simultaneously. As a result, the initial estimation of the common structure is less dependent on subject ordering than sequential procrustes-based template construction.

The PCA-based procedure can be understood in three conceptual steps.

**Step A: Estimate a PCA-based group template.**  
Response matrices from all subjects are combined along the feature dimension, and singular value decomposition (SVD) is applied to the concatenated data. A subset of principal components is retained to form a low-dimensional template, denoted as $M_{\mathrm{PC}}$, which captures the dominant response structure shared across subjects.

**Step B: Express the PCA template in the original feature space.**  
The PCA-derived template is initially defined in a reduced component space. To make it compatible with subject-level voxel or vertex data, the template can be mapped back to the original feature space. In this step, an orthogonal procrustes transformation is estimated to best match the PCA-derived template to the subject data:

$$
R =
\arg\min_{R}
\sum_{p=1}^{N}
\left\|
M_{\mathrm{PC}} R - B_p
\right\|_F^2
\quad
\text{subject to}
\quad
R^\top R = I .
$$

Here, $B_p \in \mathbb{R}^{T \times V}$ denotes the response matrix of subject $p$, where $T$ is the number of matched time points or stimulus samples and $V$ is the number of voxels or vertices. In this context, procrustes rotation is used as a projection or reconstruction step: it helps express the PCA-derived shared structure in the original feature space, rather than serving as the primary mechanism by which the common structure is estimated.

**Step C: Aggregate the projected representations.**  
After the PCA-derived template has been expressed in the feature space, the resulting representations can be aggregated to form the final common template. This template can then be used as a reference space for subsequent hyperalignment, transformation estimation, or cross-subject multivariate analyses.

Overall, the PCA-based approach provides an alternative strategy for common space construction. By estimating shared structure through a group-level decomposition, it reduces sensitivity to subject ordering while still producing a template that can be related back to the original voxel or vertex space.

## 4. Transformation Matrix (T Matrix)

After constructing a common representational space, the next step is to estimate subject-specific T matrices. These matrices define how each subject's raw response data can be mapped into the common space. In procrustes-based hyperalignment, each T matrix is estimated using the orthogonal procrustes procedure introduced in Section 2.

Once estimated, the T matrices can be applied to new or held-out fMRI data from the same subjects. This allows response patterns originally expressed in subject-specific voxel or vertex coordinates to be re-expressed in the shared common space, enabling cross-subject comparison, averaging, decoding, or other multivariate analyses.

<iframe src="alignment_pattern_align.html" title="Subjects aligning to a fixed common template demo" style="width: 100%; aspect-ratio: 900 / 800; height: auto; border: 0; display: block;" loading="lazy"></iframe>


## 5. Searchlight Hyperalignment

```{image} pic/understand_ha/sls_ha.png
:width: 800px
```

In searchlight-based hyperalignment, each searchlight defines a local response space. Within this local space, response patterns from different subjects are aligned by constructing a local common space and estimating subject-specific T matrices. 

The procedure is repeated independently for many overlapping searchlights across the brain. Because searchlights overlap, each voxel or surface vertex can belong to multiple local neighborhoods and can therefore receive multiple locally estimated transformations. Rather than selecting a single transformation, these local mappings are combined to form a voxel-wise or vertex-wise transformation. In many implementations, this aggregation is performed using distance-based weighting, so that searchlights centered closer to a given voxel or vertex contribute more strongly to its final transformation. In `fMRI-HA`, the distance weight for a voxel or vertex at distance $d$ from a searchlight center is computed from the searchlight radius $r$ as

$$w = \frac{r - d}{r}$$

These weights are then normalized across all overlapping searchlights that contain the same voxel or vertex before the local mappings are aggregated.

Through this local-to-global aggregation, searchlight hyperalignment produces a whole-brain mapping from each subject's native representational space into a shared common space. Applying the resulting transformations to fMRI data allows individual response patterns to be re-expressed in a common coordinate system, while still respecting the local structure of multivariate representations. This makes it possible to compare, average, decode, or model response patterns across individuals despite substantial variability in fine-scale cortical organization.